In [23]:
import logging
from rich.logging import RichHandler

logging.basicConfig(
    level=logging.INFO,
    format="%(message)s",
    datefmt="[%X]",
    handlers=[RichHandler(rich_tracebacks=True)],
)

LOG: logging.Logger = logging.getLogger(__name__)

In [38]:
from epicsarchiver.mgmt.archiver_mgmt_info import ArchivingStatus
from epicsarchiver.mgmt.archiver_mgmt_operations import Storage

In [42]:
def delete_multiple_pvs(archiver, pv_list):
    for pv in pv_list:
        pv_status = archiver.get_archiving_status(pv)
        if pv_status != ArchivingStatus.BeingArchived:
            LOG.info(f"PV {pv} is not being archived, skipping")
            continue
        LOG.info(f"Deleting PV {pv}, first pausing")
        pause_result = archiver.pause_pv(pv)
        LOG.info(f"Pause result: {pause_result}")
        delete_result = archiver.delete_pv(pv)
        LOG.info(f"Delete result: {delete_result}")
    LOG.info("Creating status summary")
    # summary status
    for pv in pv_list:
        pv_status = archiver.get_archiving_status(pv)
        LOG.info(f"PV status: pv: {pv_status}")

In [28]:
def pause_multiple_pvs(archiver, pv_list):
    for pv in pv_list:
        LOG.info(f"Pausing PV: {pv}")
        pause_result = archiver.pause_pv(pv)
        LOG.info(f"Pause result: {pause_result}")

In [30]:
from epicsarchiver.mgmt.archiver_mgmt_info import ArchivingStatus
from epicsarchiver.mgmt.archiver_mgmt_operations import Storage

def pause_all_pvs(archiver, pv_list: list[tuple[str, str]]) -> None:
    pause_multiple_pvs(archiver, [old_pv for old_pv, _ in pv_list] + [new_pv for _, new_pv in pv_list])

def rename_multiple_pvs(archiver, pv_list: list[tuple[str, str]]) -> None:
    count = 0
    for old_pv, new_pv in pv_list:
        count += 1
        if old_pv == new_pv:
            LOG.info(f"Skipping {old_pv} to {new_pv}, already the same")
            continue
        LOG.info(f"Renaming {old_pv} to {new_pv}, {count} of {len(pv_list)}")
        new_pv_status = archiver.get_archiving_status(new_pv)
        LOG.info(f"New PV status: {new_pv_status}")
        if new_pv_status == ArchivingStatus.Paused:
            archiver.rename_and_append(old_pv, new_pv, Storage.MTS)
            LOG.info(f"Renamed {old_pv} to {new_pv}")
        else:
            archiver.pause_rename_resume_pv(old_pv, new_pv)
            LOG.info(f"Renamed {old_pv} to {new_pv}")

def resume_all_pvs(archiver, pv_list: list[tuple[str, str]]) -> None:
    # resume all the pvs
    problematic_pvs = []
    for old_pv, new_pv in pv_list:
        new_pv_status = archiver.get_archiving_status(new_pv)
        if new_pv_status == ArchivingStatus.Paused:
            LOG.info(f"Resuming {new_pv}")
            archiver.resume_pv(new_pv)
        else:
            problematic_pvs.append({new_pv: {"status": new_pv_status, "old_pv": old_pv}})
    LOG.info(f"problematic_pvs {problematic_pvs}")
    return problematic_pvs

def get_status_of_all_pvs(archiver, pv_list: list[tuple[str, str]]) -> None:
    # summary status
    for old_pv, new_pv in pv_list:
        if old_pv == new_pv:
            LOG.info(f"Skipping {old_pv} to {new_pv}, already the same")
            continue
        new_pv_status = archiver.get_archiving_status(new_pv)
        old_pv_status = archiver.get_archiving_status(old_pv)
        LOG.info(f"PV status: new_pv: {new_pv_status}, old_pv: {old_pv_status}")

### PVs

In [1]:
from pathlib import Path

hbl_files = [f for f in Path("AS-243").iterdir() if f.is_file()]

In [16]:
import csv

actions = ["Pause", "Delete", "Rename"]

def read_file(filename) -> dict[str, list[tuple[str, str]]]:
    """Reads a file, returns action, plus PVs needing actions"""
    output = {action: [] for action in actions}
    csv_data = csv.reader(open(filename), delimiter=';')
    for row in csv_data:
        if row[1] in actions[0:2]:
            output[row[1]].append((row[0].split()[0], row[1]))
        else:
            output[actions[2]].append((row[0], row[1]))
    return output




In [22]:
hbl_data = {f.name[-7:-4]: read_file(f) for f in hbl_files}

In [24]:
from epicsarchiver import ArchiverAppliance

In [25]:
archiver_linac_tn_04 = ArchiverAppliance("archiver-linac-04.tn.esss.lu.se")

### Pause PVs

#### 010

In [31]:
pause_multiple_pvs(archiver_linac_tn_04, hbl_data["010"]["Pause"])

[14:51:16] INFO     Pausing PV: ('HBL-010Crm:SC-FSM-700:VGP_1000_HI', 'Pause')                      ]8;id=400478;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=128981;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#3\3]8;;\

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV          ]8;id=321069;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=592085;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#5\5]8;;\
                    HBL-010Crm:SC-FSM-700:VGP_1000_HI', 'engine_pvName':                                           
                    'HBL-010Crm:SC-FSM-700:VGP_1000_HI', 'engine_status': 'ok', 'etl_status': 'ok',                
                    'etl_desc': 'Successfully removed PV HBL-010Crm:SC-FSM-700:VGP_1000_HI from the                
                    cluster', 'etl_pvName': 'HBL-010Crm:SC-FSM-700:VGP_1000_HI', 'status': 'ok'}                   

           INFO     Pausing PV: ('HBL-010Crm:SC-FSM-700:VGP_1000_LO', 'Pause')                      ]8;id=440472;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=758234;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#3\3]8;;\

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV          ]8;id=545605;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=77945;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#5\5]8;;\
                    HBL-010Crm:SC-FSM-700:VGP_1000_LO', 'engine_pvName':                                           
                    'HBL-010Crm:SC-FSM-700:VGP_1000_LO', 'engine_status': 'ok', 'etl_status': 'ok',                
                    'etl_desc': 'Successfully removed PV HBL-010Crm:SC-FSM-700:VGP_1000_LO from the                
                    cluster', 'etl_pvName': 'HBL-010Crm:SC-FSM-700:VGP_1000_LO', 'status': 'ok'}                   

           INFO     Pausing PV: ('HBL-010Crm:SC-FSM-700:VGP_2000_HI', 'Pause')                      ]8;id=224877;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=298739;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#3\3]8;;\

...

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV          ]8;id=939349;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=30081;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#5\5]8;;\
                    HBL-010Crm:Cryo-EH-043:PID_DIF', 'engine_pvName':                                              
                    'HBL-010Crm:Cryo-EH-043:PID_DIF', 'engine_status': 'ok', 'etl_status': 'ok',                   
                    'etl_desc': 'Successfully removed PV HBL-010Crm:Cryo-EH-043:PID_DIF from the                   
                    cluster', 'etl_pvName': 'HBL-010Crm:Cryo-EH-043:PID_DIF', 'status': 'ok'}                      

           INFO     Pausing PV: ('HBL-010Crm:Cryo-EH-043:PV', 'Pause')                              ]8;id=617540;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=32939;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#3\3]8;;\

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV          ]8;id=939287;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=630345;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#5\5]8;;\
                    HBL-010Crm:Cryo-EH-043:PV', 'engine_pvName': 'HBL-010Crm:Cryo-EH-043:PV',                      
                    'engine_status': 'ok', 'etl_status': 'ok', 'etl_desc': 'Successfully removed PV                
                    HBL-010Crm:Cryo-EH-043:PV from the cluster', 'etl_pvName':                                     
                    'HBL-010Crm:Cryo-EH-043:PV', 'status': 'ok'}                                                   

           INFO     Pausing PV: ('HBL-010Crm:Cryo-EH-043:MAN_SP', 'Pause')                          ]8;id=56353;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=441710;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#3\3]8;;\

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV          ]8;id=42906;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=190870;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#5\5]8;;\
                    HBL-010Crm:Cryo-EH-043:MAN_SP', 'engine_pvName':                                               
                    'HBL-010Crm:Cryo-EH-043:MAN_SP', 'engine_status': 'ok', 'etl_status': 'ok',                    
                    'etl_desc': 'Successfully removed PV HBL-010Crm:Cryo-EH-043:MAN_SP from the                    
                    cluster', 'etl_pvName': 'HBL-010Crm:Cryo-EH-043:MAN_SP', 'status': 'ok'}                       

#### 020

In [32]:
pause_multiple_pvs(archiver_linac_tn_04, hbl_data["020"]["Pause"])

[14:52:22] INFO     Pausing PV: ('HBL-020Crm:SC-FSM-700:VGP_1000_HI', 'Pause')                      ]8;id=403933;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=227352;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#3\3]8;;\

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV          ]8;id=915676;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=369049;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#5\5]8;;\
                    HBL-020Crm:SC-FSM-700:VGP_1000_HI', 'engine_pvName':                                           
                    'HBL-020Crm:SC-FSM-700:VGP_1000_HI', 'engine_status': 'ok', 'etl_status': 'ok',                
                    'etl_desc': 'Successfully removed PV HBL-020Crm:SC-FSM-700:VGP_1000_HI from the                
                    cluster', 'etl_pvName': 'HBL-020Crm:SC-FSM-700:VGP_1000_HI', 'status': 'ok'}                   

           INFO     Pausing PV: ('HBL-020Crm:SC-FSM-700:VGP_1000_LO', 'Pause')                      ]8;id=315848;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=451430;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#3\3]8;;\

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV          ]8;id=456653;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=920326;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#5\5]8;;\
                    HBL-020Crm:SC-FSM-700:VGP_1000_LO', 'engine_pvName':                                           
                    'HBL-020Crm:SC-FSM-700:VGP_1000_LO', 'engine_status': 'ok', 'etl_status': 'ok',                
                    'etl_desc': 'Successfully removed PV HBL-020Crm:SC-FSM-700:VGP_1000_LO from the                
                    cluster', 'etl_pvName': 'HBL-020Crm:SC-FSM-700:VGP_1000_LO', 'status': 'ok'}                   

           INFO     Pausing PV: ('HBL-020Crm:SC-FSM-700:VGP_2000_HI', 'Pause')                      ]8;id=397371;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=249260;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#3\3]8;;\

...

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV          ]8;id=672380;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=160229;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#5\5]8;;\
                    HBL-020Crm:Cryo-EH-043:PID_DIF', 'engine_pvName':                                              
                    'HBL-020Crm:Cryo-EH-043:PID_DIF', 'engine_status': 'ok', 'etl_status': 'ok',                   
                    'etl_desc': 'Successfully removed PV HBL-020Crm:Cryo-EH-043:PID_DIF from the                   
                    cluster', 'etl_pvName': 'HBL-020Crm:Cryo-EH-043:PID_DIF', 'status': 'ok'}                      

           INFO     Pausing PV: ('HBL-020Crm:Cryo-EH-043:PV', 'Pause')                              ]8;id=928266;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=828777;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#3\3]8;;\

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV          ]8;id=743913;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=152145;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#5\5]8;;\
                    HBL-020Crm:Cryo-EH-043:PV', 'engine_pvName': 'HBL-020Crm:Cryo-EH-043:PV',                      
                    'engine_status': 'ok', 'etl_status': 'ok', 'etl_desc': 'Successfully removed PV                
                    HBL-020Crm:Cryo-EH-043:PV from the cluster', 'etl_pvName':                                     
                    'HBL-020Crm:Cryo-EH-043:PV', 'status': 'ok'}                                                   

           INFO     Pausing PV: ('HBL-020Crm:Cryo-EH-043:MAN_SP', 'Pause')                          ]8;id=153732;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=356393;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#3\3]8;;\

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV          ]8;id=186736;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=987903;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#5\5]8;;\
                    HBL-020Crm:Cryo-EH-043:MAN_SP', 'engine_pvName':                                               
                    'HBL-020Crm:Cryo-EH-043:MAN_SP', 'engine_status': 'ok', 'etl_status': 'ok',                    
                    'etl_desc': 'Successfully removed PV HBL-020Crm:Cryo-EH-043:MAN_SP from the                    
                    cluster', 'etl_pvName': 'HBL-020Crm:Cryo-EH-043:MAN_SP', 'status': 'ok'}                       

#### 030

In [33]:
pause_multiple_pvs(archiver_linac_tn_04, hbl_data["030"]["Pause"])

[14:52:28] INFO     Pausing PV: ('HBL-030Crm:SC-FSM-700:VGP_1000_HI', 'Pause')                      ]8;id=10474;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=363334;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#3\3]8;;\

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV          ]8;id=962590;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=560042;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#5\5]8;;\
                    HBL-030Crm:SC-FSM-700:VGP_1000_HI', 'engine_pvName':                                           
                    'HBL-030Crm:SC-FSM-700:VGP_1000_HI', 'engine_status': 'ok', 'etl_status': 'ok',                
                    'etl_desc': 'Successfully removed PV HBL-030Crm:SC-FSM-700:VGP_1000_HI from the                
                    cluster', 'etl_pvName': 'HBL-030Crm:SC-FSM-700:VGP_1000_HI', 'status': 'ok'}                   

           INFO     Pausing PV: ('HBL-030Crm:SC-FSM-700:VGP_1000_LO', 'Pause')                      ]8;id=755841;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=199419;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#3\3]8;;\

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV          ]8;id=741471;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=613472;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#5\5]8;;\
                    HBL-030Crm:SC-FSM-700:VGP_1000_LO', 'engine_pvName':                                           
                    'HBL-030Crm:SC-FSM-700:VGP_1000_LO', 'engine_status': 'ok', 'etl_status': 'ok',                
                    'etl_desc': 'Successfully removed PV HBL-030Crm:SC-FSM-700:VGP_1000_LO from the                
                    cluster', 'etl_pvName': 'HBL-030Crm:SC-FSM-700:VGP_1000_LO', 'status': 'ok'}                   

           INFO     Pausing PV: ('HBL-030Crm:SC-FSM-700:VGP_2000_HI', 'Pause')                      ]8;id=7264;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=262759;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#3\3]8;;\

...

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV          ]8;id=364458;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=78718;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#5\5]8;;\
                    HBL-030Crm:Cryo-EH-043:PID_DIF', 'engine_pvName':                                              
                    'HBL-030Crm:Cryo-EH-043:PID_DIF', 'engine_status': 'ok', 'etl_status': 'ok',                   
                    'etl_desc': 'Successfully removed PV HBL-030Crm:Cryo-EH-043:PID_DIF from the                   
                    cluster', 'etl_pvName': 'HBL-030Crm:Cryo-EH-043:PID_DIF', 'status': 'ok'}                      

           INFO     Pausing PV: ('HBL-030Crm:Cryo-EH-043:PV', 'Pause')                              ]8;id=930793;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=900423;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#3\3]8;;\

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV          ]8;id=51995;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=254081;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#5\5]8;;\
                    HBL-030Crm:Cryo-EH-043:PV', 'engine_pvName': 'HBL-030Crm:Cryo-EH-043:PV',                      
                    'engine_status': 'ok', 'etl_status': 'ok', 'etl_desc': 'Successfully removed PV                
                    HBL-030Crm:Cryo-EH-043:PV from the cluster', 'etl_pvName':                                     
                    'HBL-030Crm:Cryo-EH-043:PV', 'status': 'ok'}                                                   

           INFO     Pausing PV: ('HBL-030Crm:Cryo-EH-043:MAN_SP', 'Pause')                          ]8;id=49536;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=344405;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#3\3]8;;\

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV          ]8;id=446646;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=613560;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#5\5]8;;\
                    HBL-030Crm:Cryo-EH-043:MAN_SP', 'engine_pvName':                                               
                    'HBL-030Crm:Cryo-EH-043:MAN_SP', 'engine_status': 'ok', 'etl_status': 'ok',                    
                    'etl_desc': 'Successfully removed PV HBL-030Crm:Cryo-EH-043:MAN_SP from the                    
                    cluster', 'etl_pvName': 'HBL-030Crm:Cryo-EH-043:MAN_SP', 'status': 'ok'}                       

#### 040

In [34]:
pause_multiple_pvs(archiver_linac_tn_04, hbl_data["040"]["Pause"])

[14:52:32] INFO     Pausing PV: ('HBL-040Crm:SC-FSM-700:VGP_1000_HI', 'Pause')                      ]8;id=609403;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=835058;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#3\3]8;;\

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV          ]8;id=903609;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=624029;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#5\5]8;;\
                    HBL-040Crm:SC-FSM-700:VGP_1000_HI', 'engine_pvName':                                           
                    'HBL-040Crm:SC-FSM-700:VGP_1000_HI', 'engine_status': 'ok', 'etl_status': 'ok',                
                    'etl_desc': 'Successfully removed PV HBL-040Crm:SC-FSM-700:VGP_1000_HI from the                
                    cluster', 'etl_pvName': 'HBL-040Crm:SC-FSM-700:VGP_1000_HI', 'status': 'ok'}                   

           INFO     Pausing PV: ('HBL-040Crm:SC-FSM-700:VGP_1000_LO', 'Pause')                      ]8;id=227616;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=434637;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#3\3]8;;\

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV          ]8;id=7467;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=913072;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#5\5]8;;\
                    HBL-040Crm:SC-FSM-700:VGP_1000_LO', 'engine_pvName':                                           
                    'HBL-040Crm:SC-FSM-700:VGP_1000_LO', 'engine_status': 'ok', 'etl_status': 'ok',                
                    'etl_desc': 'Successfully removed PV HBL-040Crm:SC-FSM-700:VGP_1000_LO from the                
                    cluster', 'etl_pvName': 'HBL-040Crm:SC-FSM-700:VGP_1000_LO', 'status': 'ok'}                   

           INFO     Pausing PV: ('HBL-040Crm:SC-FSM-700:VGP_2000_HI', 'Pause')                      ]8;id=848342;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=973050;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#3\3]8;;\

...

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV          ]8;id=765841;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=46553;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#5\5]8;;\
                    HBL-040Crm:Cryo-EH-043:PID_DIF', 'engine_pvName':                                              
                    'HBL-040Crm:Cryo-EH-043:PID_DIF', 'engine_status': 'ok', 'etl_status': 'ok',                   
                    'etl_desc': 'Successfully removed PV HBL-040Crm:Cryo-EH-043:PID_DIF from the                   
                    cluster', 'etl_pvName': 'HBL-040Crm:Cryo-EH-043:PID_DIF', 'status': 'ok'}                      

           INFO     Pausing PV: ('HBL-040Crm:Cryo-EH-043:PV', 'Pause')                              ]8;id=381812;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=840836;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#3\3]8;;\

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV          ]8;id=486696;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=123373;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#5\5]8;;\
                    HBL-040Crm:Cryo-EH-043:PV', 'engine_pvName': 'HBL-040Crm:Cryo-EH-043:PV',                      
                    'engine_status': 'ok', 'etl_status': 'ok', 'etl_desc': 'Successfully removed PV                
                    HBL-040Crm:Cryo-EH-043:PV from the cluster', 'etl_pvName':                                     
                    'HBL-040Crm:Cryo-EH-043:PV', 'status': 'ok'}                                                   

           INFO     Pausing PV: ('HBL-040Crm:Cryo-EH-043:MAN_SP', 'Pause')                          ]8;id=281583;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=397243;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#3\3]8;;\

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV          ]8;id=310718;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py\1897436361.py]8;;\:]8;id=204576;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/1897436361.py#5\5]8;;\
                    HBL-040Crm:Cryo-EH-043:MAN_SP', 'engine_pvName':                                               
                    'HBL-040Crm:Cryo-EH-043:MAN_SP', 'engine_status': 'ok', 'etl_status': 'ok',                    
                    'etl_desc': 'Successfully removed PV HBL-040Crm:Cryo-EH-043:MAN_SP from the                    
                    cluster', 'etl_pvName': 'HBL-040Crm:Cryo-EH-043:MAN_SP', 'status': 'ok'}                       

### Delete PVs

#### 010

In [35]:
delete_multiple_pvs(archiver_linac_tn_04, hbl_data["010"]["Delete"])

[14:53:50] INFO     Deleting PV ('HBL-010Crm:SC-FSM-701:VGP_1000_HI', 'Delete'), first pausing       ]8;id=574187;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/512683969.py\512683969.py]8;;\:]8;id=389167;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/512683969.py#3\3]8;;\

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV           ]8;id=895316;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/512683969.py\512683969.py]8;;\:]8;id=752001;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/512683969.py#5\5]8;;\
                    HBL-010Crm:SC-FSM-701:VGP_1000_HI', 'engine_pvName':                                           
                    'HBL-010Crm:SC-FSM-701:VGP_1000_HI', 'engine_status': 'ok', 'etl_status': 'ok',                
                    'etl_desc': 'Successfully removed PV HBL-010Crm:SC-FSM-701:VGP_1000_HI from the                
                    cluster', 'etl_pvName': 'HBL-010Crm:SC-FSM-701:VGP_1000_HI', 'status': 'ok'}                   

           INFO     Deleting PV ('HBL-010Crm:SC-FSM-701:VGP_1000_LO', 'Delete'), first pausing       ]8;id=271653;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/512683969.py\512683969.py]8;;\:]8;id=2938;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/512683969.py#3\3]8;;\

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV           ]8;id=151428;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/512683969.py\512683969.py]8;;\:]8;id=135260;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/512683969.py#5\5]8;;\
                    HBL-010Crm:SC-FSM-701:VGP_1000_LO', 'engine_pvName':                                           
                    'HBL-010Crm:SC-FSM-701:VGP_1000_LO', 'engine_status': 'ok', 'etl_status': 'ok',                
                    'etl_desc': 'Successfully removed PV HBL-010Crm:SC-FSM-701:VGP_1000_LO from the                
                    cluster', 'etl_pvName': 'HBL-010Crm:SC-FSM-701:VGP_1000_LO', 'status': 'ok'}                   

           INFO     Deleting PV ('HBL-010Crm:SC-FSM-701:VGP_2000_HI', 'Delete'), first pausing       ]8;id=720379;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/512683969.py\512683969.py]8;;\:]8;id=668230;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/512683969.py#3\3]8;;\

...

           INFO     PV status: pv: ArchivingStatus.NotBeingArchived                                 ]8;id=819279;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/512683969.py\512683969.py]8;;\:]8;id=679883;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/512683969.py#13\13]8;;\

           INFO     PV status: pv: ArchivingStatus.NotBeingArchived                                 ]8;id=10240;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/512683969.py\512683969.py]8;;\:]8;id=156528;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/512683969.py#13\13]8;;\

           INFO     PV status: pv: ArchivingStatus.NotBeingArchived                                 ]8;id=644644;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/512683969.py\512683969.py]8;;\:]8;id=705803;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/512683969.py#13\13]8;;\

           INFO     PV status: pv: ArchivingStatus.NotBeingArchived                                 ]8;id=80894;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/512683969.py\512683969.py]8;;\:]8;id=650897;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/512683969.py#13\13]8;;\

           INFO     PV status: pv: ArchivingStatus.NotBeingArchived                                 ]8;id=491525;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/512683969.py\512683969.py]8;;\:]8;id=332443;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/512683969.py#13\13]8;;\

#### 020

In [43]:
delete_multiple_pvs(archiver_linac_tn_04, hbl_data["020"]["Delete"])

[15:44:42] INFO     Deleting PV ('HBL-020Crm:SC-FSM-701:VGP_1000_HI', 'Delete'), first pausing      ]8;id=625649;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=810958;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#7\7]8;;\

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV          ]8;id=376196;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=533947;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#9\9]8;;\
                    HBL-020Crm:SC-FSM-701:VGP_1000_HI', 'engine_pvName':                                           
                    'HBL-020Crm:SC-FSM-701:VGP_1000_HI', 'engine_status': 'ok', 'etl_status': 'ok',                
                    'etl_desc': 'Successfully removed PV HBL-020Crm:SC-FSM-701:VGP_1000_HI from the                
                    cluster', 'etl_pvName': 'HBL-020Crm:SC-FSM-701:VGP_1000_HI', 'status': 'ok'}                   

[15:44:47] INFO     Delete result: {'Engine end': 'Jan/27/2025 15:44:43 +01:00', 'Engine start':   ]8;id=98220;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=776056;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#11\11]8;;\
                    'Jan/27/2025 15:44:43 +01:00', 'Done removing PV from cluster': 'Jan/27/2025                   
                    15:44:43 +01:00', 'Start removing aliases from cluster': 'Jan/27/2025 15:44:43                 
                    +01:00', 'ETL start': 'Jan/27/2025 15:44:43 +01:00', 'Done removing aliases                    
                    from cluster': 'Jan/27/2025 15:44:47 +01:00', 'ETL end': 'Jan/27/2025 15:44:43                 
                    +01:00', 'Start removing PV from cluster': 'Jan/27/2025 15:44:43 +01:00',                      
                    'status': 'ok', 'desc': 'Successfully removed PV                                               
                    HBL-020Crm:SC-FSM-701:VGP_1000_HI from the cluster'}                                           

           INFO     Deleting PV ('HBL-020Crm:SC-FSM-701:VGP_1000_LO', 'Delete'), first pausing      ]8;id=917953;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=86501;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#7\7]8;;\

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV          ]8;id=509681;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=114392;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#9\9]8;;\
                    HBL-020Crm:SC-FSM-701:VGP_1000_LO', 'engine_pvName':                                           
                    'HBL-020Crm:SC-FSM-701:VGP_1000_LO', 'engine_status': 'ok', 'etl_status': 'ok',                
                    'etl_desc': 'Successfully removed PV HBL-020Crm:SC-FSM-701:VGP_1000_LO from the                
                    cluster', 'etl_pvName': 'HBL-020Crm:SC-FSM-701:VGP_1000_LO', 'status': 'ok'}                   

...

           INFO     PV status: pv: ArchivingStatus.NotBeingArchived                                ]8;id=502130;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=310950;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#16\16]8;;\

           INFO     PV status: pv: ArchivingStatus.NotBeingArchived                                ]8;id=340632;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=785215;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#16\16]8;;\

           INFO     PV status: pv: ArchivingStatus.NotBeingArchived                                ]8;id=581388;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=410706;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#16\16]8;;\

           INFO     PV status: pv: ArchivingStatus.NotBeingArchived                                ]8;id=237478;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=857267;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#16\16]8;;\

           INFO     PV status: pv: ArchivingStatus.NotBeingArchived                                ]8;id=609916;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=663374;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#16\16]8;;\

#### 030

In [44]:
delete_multiple_pvs(archiver_linac_tn_04, hbl_data["030"]["Delete"])

           INFO     Deleting PV ('HBL-030Crm:SC-FSM-701:VGP_1000_HI', 'Delete'), first pausing      ]8;id=447280;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=962817;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#7\7]8;;\

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV          ]8;id=325672;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=83330;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#9\9]8;;\
                    HBL-030Crm:SC-FSM-701:VGP_1000_HI', 'engine_pvName':                                           
                    'HBL-030Crm:SC-FSM-701:VGP_1000_HI', 'engine_status': 'ok', 'etl_status': 'ok',                
                    'etl_desc': 'Successfully removed PV HBL-030Crm:SC-FSM-701:VGP_1000_HI from the                
                    cluster', 'etl_pvName': 'HBL-030Crm:SC-FSM-701:VGP_1000_HI', 'status': 'ok'}                   

[15:56:23] INFO     Delete result: {'Engine end': 'Jan/27/2025 15:56:18 +01:00', 'Engine start':   ]8;id=438718;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=22648;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#11\11]8;;\
                    'Jan/27/2025 15:56:18 +01:00', 'Done removing PV from cluster': 'Jan/27/2025                   
                    15:56:18 +01:00', 'Start removing aliases from cluster': 'Jan/27/2025 15:56:18                 
                    +01:00', 'ETL start': 'Jan/27/2025 15:56:18 +01:00', 'Done removing aliases                    
                    from cluster': 'Jan/27/2025 15:56:23 +01:00', 'ETL end': 'Jan/27/2025 15:56:18                 
                    +01:00', 'Start removing PV from cluster': 'Jan/27/2025 15:56:18 +01:00',                      
                    'status': 'ok', 'desc': 'Successfully removed PV                                               
                    HBL-030Crm:SC-FSM-701:VGP_1000_HI from the cluster'}                                           

           INFO     Deleting PV ('HBL-030Crm:SC-FSM-701:VGP_1000_LO', 'Delete'), first pausing      ]8;id=536470;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=539640;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#7\7]8;;\

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV          ]8;id=150987;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=224551;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#9\9]8;;\
                    HBL-030Crm:SC-FSM-701:VGP_1000_LO', 'engine_pvName':                                           
                    'HBL-030Crm:SC-FSM-701:VGP_1000_LO', 'engine_status': 'ok', 'etl_status': 'ok',                
                    'etl_desc': 'Successfully removed PV HBL-030Crm:SC-FSM-701:VGP_1000_LO from the                
                    cluster', 'etl_pvName': 'HBL-030Crm:SC-FSM-701:VGP_1000_LO', 'status': 'ok'}                   

...

           INFO     PV status: pv: ArchivingStatus.NotBeingArchived                                ]8;id=64738;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=691586;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#16\16]8;;\

           INFO     PV status: pv: ArchivingStatus.NotBeingArchived                                ]8;id=190428;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=833408;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#16\16]8;;\

           INFO     PV status: pv: ArchivingStatus.NotBeingArchived                                ]8;id=497058;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=663512;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#16\16]8;;\

           INFO     PV status: pv: ArchivingStatus.NotBeingArchived                                ]8;id=26968;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=705970;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#16\16]8;;\

           INFO     PV status: pv: ArchivingStatus.NotBeingArchived                                ]8;id=422626;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=113804;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#16\16]8;;\

#### 040

In [45]:
delete_multiple_pvs(archiver_linac_tn_04, hbl_data["040"]["Delete"])

           INFO     Deleting PV ('HBL-040Crm:SC-FSM-701:VGP_1000_HI', 'Delete'), first pausing      ]8;id=111343;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=854154;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#7\7]8;;\

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV          ]8;id=935634;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=236567;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#9\9]8;;\
                    HBL-040Crm:SC-FSM-701:VGP_1000_HI', 'engine_pvName':                                           
                    'HBL-040Crm:SC-FSM-701:VGP_1000_HI', 'engine_status': 'ok', 'etl_status': 'ok',                
                    'etl_desc': 'Successfully removed PV HBL-040Crm:SC-FSM-701:VGP_1000_HI from the                
                    cluster', 'etl_pvName': 'HBL-040Crm:SC-FSM-701:VGP_1000_HI', 'status': 'ok'}                   

[15:59:31] INFO     Delete result: {'Engine end': 'Jan/27/2025 15:59:23 +01:00', 'Engine start':   ]8;id=693031;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=171222;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#11\11]8;;\
                    'Jan/27/2025 15:59:23 +01:00', 'Done removing PV from cluster': 'Jan/27/2025                   
                    15:59:23 +01:00', 'Start removing aliases from cluster': 'Jan/27/2025 15:59:23                 
                    +01:00', 'ETL start': 'Jan/27/2025 15:59:23 +01:00', 'Done removing aliases                    
                    from cluster': 'Jan/27/2025 15:59:31 +01:00', 'ETL end': 'Jan/27/2025 15:59:23                 
                    +01:00', 'Start removing PV from cluster': 'Jan/27/2025 15:59:23 +01:00',                      
                    'status': 'ok', 'desc': 'Successfully removed PV                                               
                    HBL-040Crm:SC-FSM-701:VGP_1000_HI from the cluster'}                                           

           INFO     Deleting PV ('HBL-040Crm:SC-FSM-701:VGP_1000_LO', 'Delete'), first pausing      ]8;id=219008;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=627754;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#7\7]8;;\

           INFO     Pause result: {'engine_desc': 'Successfully paused the archiving of PV          ]8;id=162677;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=542218;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#9\9]8;;\
                    HBL-040Crm:SC-FSM-701:VGP_1000_LO', 'engine_pvName':                                           
                    'HBL-040Crm:SC-FSM-701:VGP_1000_LO', 'engine_status': 'ok', 'etl_status': 'ok',                
                    'etl_desc': 'Successfully removed PV HBL-040Crm:SC-FSM-701:VGP_1000_LO from the                
                    cluster', 'etl_pvName': 'HBL-040Crm:SC-FSM-701:VGP_1000_LO', 'status': 'ok'}                   

...

           INFO     PV status: pv: ArchivingStatus.NotBeingArchived                                ]8;id=555020;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=358288;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#16\16]8;;\

           INFO     PV status: pv: ArchivingStatus.NotBeingArchived                                ]8;id=945214;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=621419;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#16\16]8;;\

           INFO     PV status: pv: ArchivingStatus.NotBeingArchived                                ]8;id=866848;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=712221;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#16\16]8;;\

           INFO     PV status: pv: ArchivingStatus.NotBeingArchived                                ]8;id=285412;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=205976;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#16\16]8;;\

           INFO     PV status: pv: ArchivingStatus.NotBeingArchived                                ]8;id=538887;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py\2412266754.py]8;;\:]8;id=231881;file:///var/folders/sk/7d1qtmn94msc6w_y1xrgv0fh0000gp/T/ipykernel_95073/2412266754.py#16\16]8;;\

### Rename PVs

#### 010

In [ ]:
hbl = "010"

In [ ]:
pause_all_pvs(archiver_linac_tn_04, hbl_data[hbl]["Rename"])

In [ ]:
rename_multiple_pvs(archiver_linac_tn_04, hbl_data[hbl]["Rename"])

In [ ]:
resume_all_pvs(archiver_linac_tn_04, hbl_data[hbl]["Rename"])

In [ ]:
get_status_of_all_pvs(archiver_linac_tn_04, hbl_data[hbl]["Rename"])

#### 020

In [ ]:
hbl = "020"

In [ ]:
pause_all_pvs(archiver_linac_tn_04, hbl_data[hbl]["Rename"])

In [ ]:
rename_multiple_pvs(archiver_linac_tn_04, hbl_data[hbl]["Rename"])

In [ ]:
resume_all_pvs(archiver_linac_tn_04, hbl_data[hbl]["Rename"])

In [ ]:
get_status_of_all_pvs(archiver_linac_tn_04, hbl_data[hbl]["Rename"])

#### 030

In [ ]:
pause_all_pvs(archiver_linac_tn_04, hbl_data[hbl]["Rename"])

In [ ]:
rename_multiple_pvs(archiver_linac_tn_04, hbl_data[hbl]["Rename"])

In [ ]:
resume_all_pvs(archiver_linac_tn_04, hbl_data[hbl]["Rename"])

In [ ]:
get_status_of_all_pvs(archiver_linac_tn_04, hbl_data[hbl]["Rename"])

#### 040

In [ ]:
pause_all_pvs(archiver_linac_tn_04, hbl_data[hbl]["Rename"])

In [ ]:
rename_multiple_pvs(archiver_linac_tn_04, hbl_data[hbl]["Rename"])

In [ ]:
resume_all_pvs(archiver_linac_tn_04, hbl_data[hbl]["Rename"])

In [ ]:
get_status_of_all_pvs(archiver_linac_tn_04, hbl_data[hbl]["Rename"])